# Momentum Analysis — v5-aligned evaluation

This notebook is the **time-series-momentum (TSMOM) regression benchmark** for the thesis,
scored on the **identical protocol** as `model_htboost_v5.ipynb` (the source of truth) via
the shared `thesis_eval` module. On the Norwegian swap tenors it produces:

1. A **continuous out-of-sample forecast** of the h-day rate *change* from a TSMOM OLS
   regression of the h-day change on the past lookback-window return, refit on each fold's
   training window (lookback chosen by **inner validation on training rows only**).
2. The **shared metrics long-CSV** (`momentum_metrics_long.csv`) that merges with
   `v5_metrics_long.csv` with zero reconciliation.
3. A **trading-sim export** (`momentum_oos_predictions.csv`) of causal walk-forward OOS
   predictions with positions and a threshold-ready `yhat` / `train_pred_std`.

**Protocol (from `thesis_eval`, copied from v5):** walk-forward expanding-window regime
folds (GFC, ZIRP×2, COVID, Hiking); target `y = level.shift(-h) - level`; label-overlap
purge of `H-1` business days at every train/test boundary; horizons `H ∈ {1,5,21,63}`;
Random-walk benchmark (`ŷ_RW = 0`); Campbell–Thompson OOS-R², Clark–West,
Diebold–Mariano–Harvey, Pesaran–Timmermann, one-sided binomial; Bonferroni + BH-FDR within
the walk-forward `{horizon × tenor × regime}` family.

> The full-sample lag-1 autocorrelation, Ljung–Box and Hurst tests below are **descriptive
> motivation only** — computed on the whole sample, *not* evidence of OOS skill.

## References — literature motivating each part

**Momentum / trend premia (descriptive screen, model spec & trading construction).**
- Jegadeesh, N., & Titman, S. (1993). Returns to Buying Winners and Selling Losers: Implications for Stock Market Efficiency. *Journal of Finance*, 48(1), 65–91.
- Moskowitz, T. J., Ooi, Y. H., & Pedersen, L. H. (2012). Time Series Momentum. *Journal of Financial Economics*, 104(2), 228–250.
- Asness, C. S., Moskowitz, T. J., & Pedersen, L. H. (2013). Value and Momentum Everywhere. *Journal of Finance*, 68(3), 929–985.
- Hurst, B., Ooi, Y. H., & Pedersen, L. H. (2017). A Century of Evidence on Trend-Following Investing. *Journal of Portfolio Management*, 44(1), 15–29.
- Brooks, J., & Moskowitz, T. J. (2017). Yield Curve Premia. Working paper (yield-curve risk premia).

**Serial-dependence diagnostics (motivation only, not OOS evidence).**
- Ljung, G. M., & Box, G. E. P. (1978). On a Measure of Lack of Fit in Time Series Models. *Biometrika*, 65(2), 297–303.
- Hurst, H. E. (1951). Long-Term Storage Capacity of Reservoirs. *Transactions of the American Society of Civil Engineers*, 116, 770–799.
- Lo, A. W. (1991). Long-Term Memory in Stock Market Prices. *Econometrica*, 59(5), 1279–1313. — R/S rescaled-range long-memory caveat.

**Out-of-sample evaluation, multiple testing & cross-validation (shared harness).**
- Campbell, J. Y., & Thompson, S. B. (2008). Predicting Excess Stock Returns Out of Sample: Can Anything Beat the Historical Average? *Review of Financial Studies*, 21(4), 1509–1531.
- Welch, I., & Goyal, A. (2008). A Comprehensive Look at the Empirical Performance of Equity Premium Prediction. *Review of Financial Studies*, 21(4), 1455–1508.
- Clark, T. E., & West, K. D. (2007). Approximately Normal Tests for Equal Predictive Accuracy in Nested Models. *Journal of Econometrics*, 138(1), 291–311.
- Diebold, F. X., & Mariano, R. S. (1995). Comparing Predictive Accuracy. *Journal of Business & Economic Statistics*, 13(3), 253–263.
- Harvey, D., Leybourne, S., & Newbold, P. (1997). Testing the Equality of Prediction Mean Squared Errors. *International Journal of Forecasting*, 13(2), 281–291.
- Pesaran, M. H., & Timmermann, A. (1992). A Simple Nonparametric Test of Predictive Performance. *Journal of Business & Economic Statistics*, 10(4), 461–465.
- Benjamini, Y., & Hochberg, Y. (1995). Controlling the False Discovery Rate: A Practical and Powerful Approach to Multiple Testing. *Journal of the Royal Statistical Society: Series B*, 57(1), 289–300.
- Harvey, C. R., Liu, Y., & Zhu, H. (2016). ... and the Cross-Section of Expected Returns. *Review of Financial Studies*, 29(1), 5–68.
- López de Prado, M. (2018). *Advances in Financial Machine Learning*. Wiley (purged K-fold + embargo).
- Bergmeir, C., Hyndman, R. J., & Koo, B. (2018). A Note on the Validity of Cross-Validation for Evaluating Autoregressive Time Series Prediction. *Computational Statistics & Data Analysis*, 120, 70–83.

## Pre-registered decision rule (stated before running)

This is a **null-confirmation harness** (mirroring `model_htboost_v5.ipynb`). We commit,
*ex ante*, to the following standard for declaring genuine momentum skill:

1. **Multiple testing.** A `(tenor × horizon × regime)` cell counts only if its one-sided
   binomial p-value clears **Bonferroni** *or* **BH-FDR** correction (Benjamini & Hochberg,
   1995; Harvey, Liu & Zhu, 2016) over the full walk-forward family `N` — recomputed, and
   **never pooled** across the NOR and full-universe families.
2. **Multiple regimes.** Positive OOS directional accuracy must hold across **multiple**
   regimes (GFC, ZIRP, COVID, Hiking), not a single one. One-regime strength is not a lead.
3. **Null confirmation.** OOS `dir_acc ≈ 0.50` together with a **negative Campbell–Thompson
   OOS-R²** (`ct_r2_oos < 0` — the model loses to the random walk; Campbell & Thompson, 2008)
   **confirms the null**. That is the designed, expected outcome; the train-vs-OOS gap is the
   finding, not a failure.

Because the h-day labels overlap, the directional tests are inflated. We therefore report the
**effective sample size** `n_eff ≈ n/H` next to every `dir_acc`; the binomial and
Pesaran–Timmermann (1992) p-values must be read against `n_eff`, not the raw `n`.

In [ ]:
import sys, os
import warnings
import numpy as np
import pandas as pd
from statsmodels.stats.diagnostic import acorr_ljungbox

# Put the repo root (dir containing src/) on sys.path so the flat root-level module
# (thesis_eval) imports whether cwd is the repo root or notebooks/.
_ROOT = os.getcwd()
while _ROOT != os.path.dirname(_ROOT) and not os.path.isdir(os.path.join(_ROOT, 'src')):
    _ROOT = os.path.dirname(_ROOT)
if _ROOT not in sys.path:
    sys.path.insert(0, _ROOT)
from src.data.bloomberg import load_data
from src.config import MATURITY_NAMES
import thesis_eval as te

pd.set_option('display.max_rows', 120)
# Narrowed from a blanket ignore so genuine issues surface during the handoff run;
# keep only the known-benign numerical/statsmodels warnings the descriptive screens emit.
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ── Identity / provenance ────────────────────────────────────────────────────
NOTEBOOK   = 'momentum'
MODEL_KIND = 'momentum'
RUN_TS     = pd.Timestamp.now(tz='UTC').isoformat()

# ── Headline universe + horizons from the shared module (same as v5) ─────────
UNIVERSE   = te.NOR_TENORS
H_GRID     = te.H_GRID
WF_FOLDS   = te.WF_FOLDS
MIN_TRAIN_OBS, MIN_TEST_OBS = te.MIN_TRAIN_OBS, te.MIN_TEST_OBS

# ── Frozen model config + pre-committed lookback grid ────────────────────────
# TSMOM OLS:  y = a + b * past_return(lookback).  The lookback is selected by INNER
# validation on each fold's TRAINING rows only (time-ordered holdout, purged by H-1);
# the grid below is pre-committed and never tuned on OOS data.
LOOKBACKS    = [5, 10, 20, 60, 120, 250]
DEFAULT_LB   = 20                    # fallback if inner-CV cannot score
INNER_VAL_FRAC = 0.25               # last 25% of train rows = inner validation
FROZEN_CFG   = dict(model='tsmom_ols', target='level_change',
                    lookbacks=LOOKBACKS, inner_val_frac=INNER_VAL_FRAC)
CONFIG_HASH  = te.config_hash(FROZEN_CFG, extra='tsmom_ols_levelchg')
FEATURE_COUNT = 1                    # the past lookback-window return

# ── Trading-export conventions (STEP 5) ──────────────────────────────────────
# Task convention: long (+1) = fixed-rate payer; predicted RATE DECREASE -> long (+1),
# predicted increase -> short (-1)  => position = -sign(yhat).  (Literal task instruction;
# opposite sign of swap_PL.ipynb's payer convention — flip LONG_ON_DECREASE to switch.)
LONG_ON_DECREASE = True
DEFAULT_X = 0.0                      # |yhat| > X*train_pred_std gate; X=0 == v5 always-on

OUT_METRICS = 'momentum_metrics_long.csv'
OUT_PRED    = 'momentum_oos_predictions.csv'
V5_METRICS  = 'htboost_results_v5_nor/v5_metrics_long.csv'

print('Momentum (TSMOM OLS) — v5-aligned harness')
print(f'  universe = {UNIVERSE}')
print(f'  horizons = {H_GRID}   lookback grid = {LOOKBACKS}')
print(f'  folds    = {te.FOLD_NAMES}')
print(f'  config_hash = {CONFIG_HASH}  long_on_decrease={LONG_ON_DECREASE}  default_X={DEFAULT_X}')

In [ ]:
df_all = load_data()
print(f'Loaded {df_all.shape[0]} rows x {df_all.shape[1]} columns')
present = [s for s in UNIVERSE if s in df_all.columns]
missing = [s for s in UNIVERSE if s not in df_all.columns]
if missing:
    print('WARNING missing tenors:', missing)
print('\nHeadline universe coverage:')
for s in present:
    ser = df_all[s].dropna()
    print(f'  {s:<9s} n={len(ser):5d}  [{ser.index.min().date()} -> {ser.index.max().date()}]')

## Descriptive motivation (full sample) — NOT OOS evidence

Lag-1 autocorrelation of daily changes, Ljung–Box (H0: no serial correlation) and the
Hurst exponent on the **whole** sample, shown only to motivate fitting a momentum model.
These are *not* used for selection and are *not* claimed as out-of-sample skill.

*Motivation (full sample only):* lag-1 autocorrelation, the Ljung & Box (1978) statistic and the Hurst (1951) exponent (with the Lo, 1991, R/S long-memory caveat) motivate a time-series-momentum specification (Moskowitz, Ooi & Pedersen, 2012).

In [ ]:
def lag1_autocorr(s):
    s = s.dropna().diff().dropna()
    return round(float(s.autocorr(lag=1)), 4) if len(s) >= 10 else np.nan

def ljungbox_test(s, lags=10):
    s = s.dropna().diff().dropna()
    if len(s) < lags + 5: return dict(lb_stat=np.nan, lb_p=np.nan, lb_reject=np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore'); r = acorr_ljungbox(s, lags=[lags], return_df=True)
    p = float(r['lb_pvalue'].iloc[-1])
    return dict(lb_stat=round(float(r['lb_stat'].iloc[-1]),3), lb_p=round(p,4), lb_reject=bool(p<0.05))

def hurst_exponent(s, min_window=20):
    s = s.dropna().values; n = len(s)
    if n < 100: return np.nan
    max_window = n // 2
    if max_window < min_window: return np.nan
    lags = np.unique(np.logspace(np.log10(min_window), np.log10(max_window), 30).astype(int))
    rs_means, valid = [], []
    for lag in lags:
        rs = []
        for start in range(0, n - lag, lag):
            ch = s[start:start+lag]; dev = np.cumsum(ch - ch.mean())
            R = dev.max() - dev.min(); S = ch.std(ddof=1)
            if S > 0: rs.append(R/S)
        if rs: rs_means.append(np.mean(rs)); valid.append(lag)
    if len(valid) < 2: return np.nan
    return round(float(np.polyfit(np.log(valid), np.log(rs_means), 1)[0]), 4)

rows = []
for s in present:
    ser = df_all[s].dropna().sort_index()
    rows.append({'Series': s, 'n_obs': len(ser), 'lag1_autocorr': lag1_autocorr(ser),
                 **ljungbox_test(ser), 'hurst': hurst_exponent(ser)})
desc_df = pd.DataFrame(rows).set_index('Series')
print('Descriptive momentum diagnostics (full sample, motivation only):')
display(desc_df)

## Continuous TSMOM forecast — walk-forward, train + OOS

For each `(security, horizon, fold)` we choose the lookback by inner validation on the
fold's **training** rows (time-ordered: last 25% held out, purged by `H-1`), then refit the
OLS on all training rows:

```
past_return_t = level_t - level_{t-lookback}
y = a + b * past_return_t           (OLS on train)
yhat(t) = a + b * past_return_t
```

a continuous momentum forecast of the h-day change. One TRAIN and one OOS row per populated
fold are scored with `compute_metrics_row`; the chosen lookback is recorded as an extra
column.

*Model spec:* the TSMOM regression of the h-day change on the past lookback-window return follows Moskowitz, Ooi & Pedersen (2012); the lookback blend echoes Hurst, Ooi & Pedersen (2017) and the cross-asset evidence of Asness, Moskowitz & Pedersen (2013).

In [ ]:
def _ols_fit(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 5: return None
    b, a = np.polyfit(x[m], y[m], 1)   # y = a + b*x
    return float(a), float(b)

def _ols_pred(x, ab):
    if ab is None: return np.zeros_like(np.asarray(x, float))
    a, b = ab
    return a + b * np.asarray(x, float)

def select_lookback(level, train_dates, y_series, h):
    """Inner-validation lookback selection on TRAIN rows only. Returns best lookback."""
    tr = pd.DataFrame({'date': train_dates}).set_index('date')
    tr['y'] = y_series.reindex(tr.index)
    n = len(tr); cut = int(n * (1 - INNER_VAL_FRAC))
    if cut < 30 or n - cut < 10:
        return DEFAULT_LB
    in_idx  = tr.index[:cut]
    val_idx = tr.index[cut + (h - 1):]           # inner purge of H-1
    best_lb, best_mse = DEFAULT_LB, np.inf
    for lb in LOOKBACKS:
        r = (level - level.shift(lb))
        ab = _ols_fit(r.reindex(in_idx).to_numpy(float), tr['y'].reindex(in_idx).to_numpy(float))
        if ab is None: continue
        yhat = _ols_pred(r.reindex(val_idx).to_numpy(float), ab)
        yv = tr['y'].reindex(val_idx).to_numpy(float)
        m = np.isfinite(yhat) & np.isfinite(yv)
        if m.sum() < 5: continue
        mse = float(np.mean((yv[m] - yhat[m]) ** 2))
        if mse < best_mse:
            best_mse, best_lb = mse, lb
    return best_lb

def run_security_mom(df, sec, h):
    country, tenor = sec.rsplit('_', 1)
    level = df[sec].dropna().sort_index()
    y_series = level.shift(-h) - level
    panel = pd.DataFrame({'date': level.index, 'level': level.values, 'y': y_series.values})
    base = dict(notebook=NOTEBOOK, run_ts=RUN_TS, model_kind=MODEL_KIND, is_pooled=False,
                validation_scheme='walk_forward', target_kind='level_change',
                security=sec, country=country, tenor=tenor,
                config_hash=CONFIG_HASH, feature_count=FEATURE_COUNT)
    rows, preds = [], []
    for fold_name, ts, tend, regime in WF_FOLDS:
        tr, tef = te.wf_purge_split(panel, ts, tend, h)
        if len(tr) < MIN_TRAIN_OBS or len(tef) < MIN_TEST_OBS:
            continue
        lb = select_lookback(level, tr['date'].values, y_series, h)
        r = (level - level.shift(lb))
        ab = _ols_fit(r.reindex(tr['date'].values).to_numpy(float), tr['y'].to_numpy(float))
        yhat_tr = _ols_pred(r.reindex(tr['date'].values).to_numpy(float), ab)
        yhat_te = _ols_pred(r.reindex(tef['date'].values).to_numpy(float), ab)
        # train_pred_std is computed from TRAIN-fold predictions ONLY (no OOS leakage)
        train_pred_std = float(np.std(yhat_tr[np.isfinite(yhat_tr)])) if np.isfinite(yhat_tr).any() else 0.0
        meta = {**base, 'fold': fold_name, 'regime': regime}
        rtr = te.compute_metrics_row(tr['y'].to_numpy(float), yhat_tr, h, {**meta, 'sample': 'train'})
        rte = te.compute_metrics_row(tef['y'].to_numpy(float), yhat_te, h, {**meta, 'sample': 'oos'})
        rtr['lookback'] = lb; rte['lookback'] = lb
        rows.extend([rtr, rte])
        sub = pd.DataFrame({'date': tef['date'].values, 'security': sec, 'country': country,
                            'tenor': tenor, 'horizon': h, 'regime': regime,
                            'level_t': tef['level'].values,
                            'y_true': tef['y'].values, 'yhat': yhat_te,
                            'train_pred_std': train_pred_std})
        preds.append(sub)
    return rows, preds

print('Model defined: select_lookback (inner CV), run_security_mom')

In [ ]:
metric_rows, pred_parts = [], []
for h in H_GRID:
    for sec in present:
        r, p = run_security_mom(df_all, sec, h)
        metric_rows.extend(r); pred_parts.extend(p)
        oos = [x for x in r if x['sample'] == 'oos']
        da = ', '.join(f"{x['dir_acc']:.3f}" for x in oos)
        lbs = ', '.join(str(x['lookback']) for x in oos)
        print(f'  H={h:<3d} {sec:<9s} folds={len(oos)}  OOS_DA=[{da}]  lookbacks=[{lbs}]')

df_metrics = pd.DataFrame(metric_rows)
print(f'\ndf_metrics: {df_metrics.shape}  (train+oos rows)')

## Out-of-sample evaluation & multiple testing

Every fold is scored against a random-walk benchmark (`ŷ_RW = 0`) with the Campbell–Thompson
(2008) OOS-R², the Clark–West (2007) and Diebold–Mariano (1995) / Harvey–Leybourne–Newbold
(1997) equal-predictive-accuracy tests (HAC/Newey–West, lag `H−1`), the Pesaran–Timmermann
(1992) directional test, and a one-sided binomial. The walk-forward `{horizon × tenor ×
regime}` family is corrected with Bonferroni and BH-FDR (Benjamini & Hochberg, 1995; the
multiple-testing-in-finance critique of Harvey, Liu & Zhu, 2016). Walk-forward validation
follows the time-series cross-validation guidance of Bergmeir, Hyndman & Koo (2018) and the
purge/embargo discipline of López de Prado (2018).

In [ ]:
# ── Multiple-testing correction within the walk-forward family (v5 convention) ──
df_metrics = te.finalize_long_csv(df_metrics)
N, nb, nh = te.apply_mtc_family(df_metrics, ['walk_forward'], te.WF_MTC_FAMILY)
df_metrics = te.finalize_long_csv(df_metrics)
df_metrics.to_csv(OUT_METRICS, index=False)
print(f'Wrote {OUT_METRICS}: rows={len(df_metrics)} cols={df_metrics.shape[1]}')
print(f'  walk-forward MTC family: N={N}  Bonferroni survivors={nb}  BH-FDR={nh}')

## Trading-sim export

The continuous TSMOM forecast is turned into a position with the time-series-momentum sign
convention of Moskowitz, Ooi & Pedersen (2012) and Hurst, Ooi & Pedersen (2017): raw `yhat`
and the **train-only** `train_pred_std` are emitted so the downstream simulator can sweep the
volatility-scaled entry threshold without re-fitting. Carry/roll is **not** fabricated here —
the simulator joins it from the shared curve layer using `level_t` + `tenor` (Brooks &
Moskowitz, 2017).

In [ ]:
# ── Trading-sim export: causal walk-forward OOS predictions only (STEP 5) ────
pred = pd.concat(pred_parts, ignore_index=True)
pred = pred[np.isfinite(pred['level_t'])].reset_index(drop=True)   # drop missing level, never zero-fill

def positions_from(yhat, train_std, X=DEFAULT_X, long_on_decrease=LONG_ON_DECREASE):
    yhat = np.asarray(yhat, float)
    thr = X * np.asarray(train_std, float)
    direction = -1.0 if long_on_decrease else 1.0   # predicted decrease (yhat<0) -> +1
    return np.where(np.abs(yhat) > thr, direction * np.sign(yhat), 0.0)

pred['position'] = positions_from(pred['yhat'].values, pred['train_pred_std'].values)
pred = pred[['date', 'security', 'country', 'tenor', 'horizon', 'regime',
             'level_t', 'y_true', 'yhat', 'train_pred_std', 'position']]
pred = pred.sort_values(['horizon', 'date', 'security']).reset_index(drop=True)
pred.to_csv(OUT_PRED, index=False)
print(f'Wrote {OUT_PRED}: {len(pred)} OOS rows')
print('  position mix:', pred['position'].value_counts().to_dict())
print(pred.head(6).to_string(index=False))

In [ ]:
# ── Summary: OOS directional accuracy + Campbell-Thompson R2 by regime & horizon ──
oos = df_metrics[df_metrics['sample'] == 'oos'].copy()
print('=== Mean OOS directional accuracy  (regime x horizon) ===')
display(oos.pivot_table(index='regime', columns='horizon', values='dir_acc', aggfunc='mean').round(3))
print('=== Mean OOS Campbell-Thompson R2  (regime x horizon) ===')
display(oos.pivot_table(index='regime', columns='horizon', values='ct_r2_oos', aggfunc='mean').round(4))
print('=== Mean OOS n_eff (approx n/H)  (regime x horizon) — read dir_acc significance against this ===')
display(oos.pivot_table(index='regime', columns='horizon', values='n_eff', aggfunc='mean').round(0))
print('  (overlapping h-day labels inflate the binomial / Pesaran-Timmermann tests: judge them on n_eff, not n)')

n_cells = len(oos)
print(f'(security x horizon x regime) OOS cells: {n_cells}')
print(f"  survive Bonferroni: {int(oos['reject_bonferroni'].sum())}    survive BH-FDR: {int(oos['reject_fdr_bh'].sum())}")

print('\n=== Train vs OOS dir_acc by horizon (overfit gap) ===')
gap = df_metrics.pivot_table(index='horizon', columns='sample', values='dir_acc', aggfunc='mean')
gap['gap(train-oos)'] = gap.get('train') - gap.get('oos')
display(gap.round(3))

In [ ]:
# ── Merge contract: must concatenate with v5_metrics_long.csv, zero reconciliation ──
v5 = pd.read_csv(V5_METRICS)
shared = te.SHARED_COLS
assert not [c for c in shared if c not in df_metrics.columns], 'missing shared cols here'
assert not [c for c in shared if c not in v5.columns], 'missing shared cols in v5'
merged = pd.concat([v5, df_metrics], ignore_index=True)
assert all(c in merged.columns for c in shared), 'shared columns lost on concat'
print('MERGE OK — shared schema identical, no renames/drops.')
print(f'  v5 rows={len(v5)}  +  momentum rows={len(df_metrics)}  =  merged {len(merged)}')
print(f'  shared cols={len(shared)}  merged cols={merged.shape[1]}')
print('  notebooks in merged:', sorted(merged["notebook"].unique()))

## Full universe of swaps — same pipeline, separate multiple-testing family

A **separate** section from the NOR head-to-head above. It runs the *identical* causal
walk-forward pipeline (`run_security_mom`, unchanged) on every available `country × tenor`
with enough history, writes **separate** files (`momentum_metrics_long_universe.csv`,
`momentum_oos_predictions_universe.csv`), and applies its **own** multiple-testing family (the
universe grid) so it can never contaminate the NOR family. The breadth across markets is the
cross-sectional analogue of Asness, Moskowitz & Pedersen (2013); the wider grid makes the
Harvey, Liu & Zhu (2016) data-snooping correction more, not less, binding. **The NOR numbers
above are unchanged by anything in this section.**

In [ ]:
# (Universe a) definition — all swap prefixes × te tenor suffixes present with enough history
SWAP_PREFIXES  = ['AUS','BRAZ','CAN','CHIN','EUR','JPY','NEWZ','NOR','POL','SOFR','SONIA','SWE','SWZ','TURK']
TENOR_SUFFIXES = [t.split('_', 1)[1] for t in te.NOR_TENORS]      # 1Y, 5Y, 10Y, 15Y, 30Y
MIN_UNIV_OBS   = te.MIN_TRAIN_OBS + te.MIN_TEST_OBS               # same gate as a scored fold pair
universe = [f'{p}_{m}' for p in SWAP_PREFIXES for m in TENOR_SUFFIXES
            if f'{p}_{m}' in df_all.columns
            and df_all[f'{p}_{m}'].dropna().shape[0] >= MIN_UNIV_OBS]
print(f'Universe: {len(universe)} securities across {len(SWAP_PREFIXES)} markets x {TENOR_SUFFIXES}')
print(f'  inclusion gate: >= MIN_TRAIN_OBS + MIN_TEST_OBS = {MIN_UNIV_OBS} obs')
print(' ', universe)

In [ ]:
# (Universe b) descriptive motivation screen — reuse the functions defined above (no redefine)
urows = []
for c in universe:
    s = df_all[c].dropna().sort_index()
    urows.append({'Series': c, 'n_obs': len(s), 'lag1_autocorr': lag1_autocorr(s),
                  **ljungbox_test(s), 'hurst': hurst_exponent(s)})
univ_desc = pd.DataFrame(urows).set_index('Series')
univ_desc['momentum'] = (univ_desc['lag1_autocorr'] > 0) & (univ_desc['lb_reject'] == True) & (univ_desc['hurst'] > 0.5)
print(f'Universe descriptive screen ({len(univ_desc)} series, motivation only); '
      f"momentum-consistent: {int(univ_desc['momentum'].sum())}")
display(univ_desc.sort_values('hurst', ascending=False))

In [ ]:
# (Universe c) causal walk-forward sweep — reuse run_security_mom UNCHANGED over the universe
univ_rows, univ_preds = [], []
for h in H_GRID:
    for sec in universe:
        r, p = run_security_mom(df_all, sec, h)
        univ_rows.extend(r); univ_preds.extend(p)
df_univ = pd.DataFrame(univ_rows)
print(f'Universe sweep: df_univ {df_univ.shape}  ({len(universe)} secs x {len(H_GRID)} horizons)')

In [ ]:
# (Universe d) finalize + SEPARATE MTC family (the universe grid) + breakdowns + write CSV
OUT_METRICS_UNIV = 'momentum_metrics_long_universe.csv'
UNIV_MTC_FAMILY  = 'walk_forward:universe:{country x tenor x horizon x regime}'
df_univ = te.finalize_long_csv(df_univ)
Nu, nbu, nhu = te.apply_mtc_family(df_univ, ['walk_forward'], UNIV_MTC_FAMILY)
df_univ = te.finalize_long_csv(df_univ)
df_univ.to_csv(OUT_METRICS_UNIV, index=False)
print(f'Wrote {OUT_METRICS_UNIV}: rows={len(df_univ)} cols={df_univ.shape[1]}')
print(f'  universe MTC family N={Nu}  Bonferroni survivors={nbu}  BH-FDR={nhu}  '
      f'(SEPARATE family from NOR)')

uoos = df_univ[df_univ['sample'] == 'oos'].copy()
print('\n=== Universe mean OOS dir_acc  (regime x horizon) ===')
display(uoos.pivot_table(index='regime', columns='horizon', values='dir_acc', aggfunc='mean').round(3))
print('=== Universe mean OOS Campbell-Thompson R2  (regime x horizon) ===')
display(uoos.pivot_table(index='regime', columns='horizon', values='ct_r2_oos', aggfunc='mean').round(4))
print('=== Universe mean OOS dir_acc  (country x horizon) ===')
display(uoos.pivot_table(index='country', columns='horizon', values='dir_acc', aggfunc='mean').round(3))
print('=== Universe mean OOS Campbell-Thompson R2  (country x horizon) ===')
display(uoos.pivot_table(index='country', columns='horizon', values='ct_r2_oos', aggfunc='mean').round(4))
print('=== Universe mean OOS n_eff (approx n/H)  (regime x horizon) — read dir_acc significance against this ===')
display(uoos.pivot_table(index='regime', columns='horizon', values='n_eff', aggfunc='mean').round(0))
print(f'\nUniverse OOS cells: {len(uoos)}   survive Bonferroni: {int(uoos["reject_bonferroni"].sum())}'
      f'   survive BH-FDR: {int(uoos["reject_fdr_bh"].sum())}   (family N={Nu})')

In [ ]:
# (Universe e) trading export — RUN AFTER the tests above; identical schema to the NOR export
OUT_PRED_UNIV = 'momentum_oos_predictions_universe.csv'
upred = pd.concat(univ_preds, ignore_index=True)
upred = upred[np.isfinite(upred['level_t'])].reset_index(drop=True)   # drop missing level, never zero-fill
upred['position'] = positions_from(upred['yhat'].values, upred['train_pred_std'].values)
upred = upred[['date', 'security', 'country', 'tenor', 'horizon', 'regime',
               'level_t', 'y_true', 'yhat', 'train_pred_std', 'position']]
upred = upred.sort_values(['horizon', 'date', 'security']).reset_index(drop=True)
upred.to_csv(OUT_PRED_UNIV, index=False)
print(f'Wrote {OUT_PRED_UNIV}: {len(upred)} OOS rows')
assert list(upred.columns) == list(pred.columns), 'universe export schema must equal the NOR export schema'
print('  schema identical to NOR export:', list(upred.columns))
print('  position mix:', upred['position'].value_counts().to_dict())

## Appendix — legacy rule-based ±1 (secondary cross-check)

Not part of the head-to-head and **not** written to the shared CSV (the ±1 signal has no magnitude). The full-universe descriptive screen now lives in the *Full universe of swaps* section above; what remains here is the legacy rule-based TSMOM sign (±1) scored on the same walk-forward folds for the NOR tenors — directional accuracy on the **full** OOS sample, with coverage as an auxiliary column.

In [ ]:
# (b) legacy rule-based TSMOM sign (+/-1) on NOR tenors, walk-forward, OOS dir_acc (full sample)
RULE_LOOKBACK = 60   # pre-committed (no OOS tuning)
print(f'Rule-based TSMOM sign (lookback={RULE_LOOKBACK}) — SECONDARY, not in CSV')
brows = []
for h in H_GRID:
    for sec in present:
        level = df_all[sec].dropna().sort_index()
        sig = np.sign(level - level.shift(RULE_LOOKBACK))
        y_series = level.shift(-h) - level
        panel = pd.DataFrame({'date': level.index, 'y': y_series.values})
        for fold_name, ts, tend, regime in WF_FOLDS:
            _, tef = te.wf_purge_split(panel, ts, tend, h)
            if len(tef) < MIN_TEST_OBS: continue
            d = tef.set_index('date'); s = sig.reindex(d.index).fillna(0.0); y = d['y']
            m = np.isfinite(y)
            da_full = float(np.mean(np.sign(s[m]) == np.sign(y[m])))
            cov = float(np.mean(s[m] != 0))
            brows.append(dict(horizon=h, security=sec, regime=regime,
                              dir_acc_full=round(da_full,3), coverage=round(cov,3), n=int(m.sum())))
rule_df = pd.DataFrame(brows)
print('Mean OOS dir_acc (full sample) by regime x horizon:')
display(rule_df.pivot_table(index='regime', columns='horizon', values='dir_acc_full', aggfunc='mean').round(3))